# House Prices - Advanced Regression Techniques

# 1. Import Modules

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

# 2. Load Data

In [2]:
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

# 3. Outlier removal

In [3]:
def remove_outliers(df):
    df = df.copy()
    df = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))]
    return df

train = remove_outliers(train)

# 4. Ordinal encoding for quality columns

In [4]:
quality_map = {
    "Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, None: 0
}

quality_cols = [
    "ExterQual", "ExterCond", "KitchenQual", "HeatingQC",
    "BsmtQual", "BsmtCond", "FireplaceQu", "GarageQual",
    "GarageCond", "PoolQC"
]

for col in quality_cols:
    train[col] = train[col].map(quality_map)
    test[col] = test[col].map(quality_map)


# 5. Neighborhood median encoding

In [5]:
neigh_median = train.groupby("Neighborhood")["SalePrice"].median()
train["NeighborhoodMedian"] = train["Neighborhood"].map(neigh_median)
test["NeighborhoodMedian"] = test["Neighborhood"].map(neigh_median)

# 6. Feature engineering

In [6]:
def add_features(df):
    df = df.copy()

    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
    df["TotalBath"] = (
        df["FullBath"] + 0.5 * df["HalfBath"] +
        df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"]
    )
    df["TotalPorchSF"] = (
        df["OpenPorchSF"] + df["EnclosedPorch"] +
        df["3SsnPorch"] + df["ScreenPorch"]
    )
    df["TotalArea"] = df["GrLivArea"] + df["TotalBsmtSF"] + df["GarageArea"]
    df["Age"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["IsRemodeled"] = (df["YearBuilt"] != df["YearRemodAdd"]).astype(int)
    df["HasPool"] = (df["PoolArea"] > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"] > 0).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)
    df["OverallScore"] = df["OverallQual"] * df["OverallCond"]
    df["QualityScore"] = df["ExterQual"].fillna(0) + df["KitchenQual"].fillna(0) + df["HeatingQC"].fillna(0)

    return df

train = add_features(train)
test = add_features(test)


# 7. Skew correction (excluding SalePrice)

In [7]:
numeric_feats = train.select_dtypes(include=["int64", "float64"]).columns
numeric_feats = numeric_feats.drop("SalePrice")

skewed = train[numeric_feats].apply(lambda x: x.dropna().skew()).sort_values(ascending=False)
skewed_feats = skewed[skewed > 0.75].index

for col in skewed_feats:
    train[col] = np.log1p(train[col])
    test[col] = np.log1p(test[col])

# 8. One-hot encoding (train + test together)

In [8]:
train_features = train.drop("SalePrice", axis=1)

full = pd.concat([train_features, test], axis=0)

full = pd.get_dummies(full, drop_first=True)

# Optional: fix whitespace in column names for LightGBM
full.columns = full.columns.str.replace(" ", "_")

X_train = full.iloc[:len(train), :].copy()
X_test = full.iloc[len(train):, :].copy()
y_train = np.log1p(train["SalePrice"])

# Fill any remaining NaNs
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

# 9. Define base models

In [9]:
models = {
    "lasso": Lasso(alpha=0.0005, max_iter=50000),
    "ridge": Ridge(alpha=5.0),
    "elastic": ElasticNet(alpha=0.0005, l1_ratio=0.9, max_iter=50000),
    "gbr": GradientBoostingRegressor(
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8
    ),
    "xgb": xgb.XGBRegressor(
        n_estimators=2000,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ),
    "lgb": lgb.LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=32,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ),
    "cat": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        verbose=False,
        random_seed=42
    )
}


# 10. Stacking helper

In [10]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

def get_oof(model, X, y, X_test, n_splits=10, random_state=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    X = np.array(X)
    X_test = np.array(X_test)
    y = np.array(y)

    oof_train = np.zeros(len(X))
    oof_test = np.zeros(len(X_test))
    oof_test_skf = np.empty((n_splits, len(X_test)))

    for i, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        oof_train[val_idx] = model.predict(X_val)
        oof_test_skf[i, :] = model.predict(X_test)

    oof_test[:] = oof_test_skf.mean(axis=0)
    return oof_train.reshape(-1, 1), oof_test.reshape(-1, 1)


# 11. stacked features + store OOF per model

In [11]:
X_stack_list = []
X_test_stack_list = []
base_oof = {}
base_cv_scores = {}

for name, model in models.items():
    print(f"Running base model: {name}")
    oof_train, oof_test = get_oof(model, X_train, y_train, X_test)
    X_stack_list.append(oof_train)
    X_test_stack_list.append(oof_test)

    base_oof[name] = oof_train  # store OOF predictions

    rmse = np.sqrt(mean_squared_error(y_train, oof_train))
    base_cv_scores[name] = rmse
    print(f"  CV RMSE (log target) for {name}: {rmse:.5f}")

print("\nBase model CV scores:")
for name, rmse in base_cv_scores.items():
    print(f"{name}: {rmse:.5f}")

X_stack = np.concatenate(X_stack_list, axis=1)
X_test_stack = np.concatenate(X_test_stack_list, axis=1)

print("Stacked feature shapes:", X_stack.shape, X_test_stack.shape)

Running base model: lasso
  CV RMSE (log target) for lasso: 0.11327
Running base model: ridge
  CV RMSE (log target) for ridge: 0.11221
Running base model: elastic
  CV RMSE (log target) for elastic: 0.11330
Running base model: gbr
  CV RMSE (log target) for gbr: 0.11936
Running base model: xgb
  CV RMSE (log target) for xgb: 0.11763
Running base model: lgb
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4563
[LightGBM] [Info] Number of data points in the train set: 1312, number of used features: 159
[LightGBM] [Info] Start training from score 12.024654


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000668 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4554
[LightGBM] [Info] Number of data points in the train set: 1312, number of used features: 159
[LightGBM] [Info] Start training from score 12.022795


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001244 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4563
[LightGBM] [Info] Number of data points in the train set: 1312, number of used features: 159
[LightGBM] [Info] Start training from score 12.027645


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000659 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4558
[LightGBM] [Info] Number of data points in the train set: 1313, number of used features: 159
[LightGBM] [Info] Start training from score 12.024490


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000706 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4554
[LightGBM] [Info] Number of data points in the train set: 1313, number of used features: 157
[LightGBM] [Info] Start training from score 12.020247


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  CV RMSE (log target) for lgb: 0.12174
Running base model: cat
  CV RMSE (log target) for cat: 0.11123

Base model CV scores:
lasso: 0.11327
ridge: 0.11221
elastic: 0.11330
gbr: 0.11936
xgb: 0.11763
lgb: 0.12174
cat: 0.11123
Stacked feature shapes: (1458, 7) (1459, 7)


# 12. Meta-model (level-2)

In [12]:
from sklearn.linear_model import Ridge

meta_model = Ridge(alpha=1.0)
meta_model.fit(X_stack, y_train)

stack_pred_log = meta_model.predict(X_stack)
stack_rmse = np.sqrt(mean_squared_error(y_train, stack_pred_log))
print(f"\nStacked meta-model RMSE (log target, in-sample): {stack_rmse:.5f}")

stack_test_log = meta_model.predict(X_test_stack)
stack_test_pred = np.expm1(stack_test_log)


Stacked meta-model RMSE (log target, in-sample): 0.10813


# 13. Fit LightGBM and XGBoost for blending

In [13]:
models["lgb"].fit(X_train, y_train)
models["xgb"].fit(X_train, y_train)

lgb_test_log = models["lgb"].predict(X_test)
xgb_test_log = models["xgb"].predict(X_test)

lgb_test_pred = np.expm1(lgb_test_log)
xgb_test_pred = np.expm1(xgb_test_log)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001350 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4658
[LightGBM] [Info] Number of data points in the train set: 1458, number of used features: 160
[LightGBM] [Info] Start training from score 12.024015


# 14. Simple blend (no fake CV, just reasonable weights)

In [14]:
w_stack = 0.6
w_lgb = 0.2
w_xgb = 0.2

blend_pred = (
    w_stack * stack_test_pred +
    w_lgb * lgb_test_pred +
    w_xgb * xgb_test_pred
)


In [15]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": blend_pred
})

submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

Saved submission.csv
